In [1]:
import numpy as np
import pandas as pd

np.random.seed(24)

stations = ["Hooghly_A", "Hooghly_B", "Damodar_A", "Damodar_B", "Rupnarayan"]
months = pd.date_range("2025-01-01", "2025-08-01", freq="MS")

rows = []

for month in months:
    for station in stations:
        rows.append({
            "date": month,
            "station": station,
            "temperature": np.random.normal(27, 3),
            "dissolved_oxygen": np.random.normal(6, 1.2),
            "bod": np.random.normal(4, 1.5),
            "nitrate": np.random.normal(3.5, 1.5),
            "sampling_depth": np.random.uniform(0.5, 3.0)
        })

water = pd.DataFrame(rows)

# Introduce realistic missing measurements
missing_idx = np.random.choice(
    water.index,
    size=10,
    replace=False
)

water.loc[missing_idx, "dissolved_oxygen"] = np.nan
water.loc[
    np.random.choice(water.index, 6, replace=False),
    "nitrate"
] = np.nan

water.head()

,date,station,temperature,dissolved_oxygen,bod,nitrate,sampling_depth
0,2025-01-01,Hooghly_A,30.987637,5.075960,3.525579,2.013784,1.301298
1,2025-01-01,Hooghly_B,31.229921,NaN,4.119309,4.899379,1.118234
2,2025-01-01,Damodar_A,29.036414,8.267127,5.442308,NaN,2.606949
3,2025-01-01,Damodar_B,29.927131,NaN,5.889212,5.843651,1.069208
4,2025-01-01,Rupnarayan,31.178564,NaN,4.182503,5.311404,2.394495


In [ ]:
# finding null values and fill with mean
water[water['nitrate'].isna()]
water.fillna(value={
    'dissolved_oxygen': water['dissolved_oxygen'].mean(),
    'nitrate': water['nitrate'].mean()
},inplace=True)

In [ ]:
# making month a seperate column
water['month'] = water['date'].dt.month

### *1. Monthly water-quality assessment*

In [26]:
water1 = water.groupby('month').agg(
    min_diss_ox = ('dissolved_oxygen','min'),
    mean_bod = ('bod','mean'),
    mean_nitrate = ('nitrate','mean')
)
water1['pol_score'] = water1.to_numpy().sum(axis=1)
water1.sort_values(by=['min_diss_ox','mean_bod','mean_nitrate'],ascending=[True,False,False])

,min_diss_ox,mean_bod,mean_nitrate,pol_score
month,,,,
3,3.741178,4.754373,4.009155,12.504705
6,4.034295,3.240017,3.309752,10.584064
2,4.408844,4.416136,3.420544,12.245524
4,4.556828,4.039748,2.956545,11.553121
1,5.075960,4.631782,4.325902,14.033644
5,5.246975,4.261740,4.120937,13.629652
7,5.562161,3.748570,2.621585,11.932316
8,5.966585,4.720302,3.725929,14.412816


#### ***Task 1 conclusion:***
March appears to be the most concerning month in this dataset as it has lowest minimum dissolved oxygen (3.74), and relatively higher bod (4.75) and nitrate concentration (4.01).

### *2. Identify consistently polluted stations*

In [28]:
water.groupby('station').agg(
    mean_bod = ('bod','mean'),
    median_nitrate = ('nitrate','median'),
    std_diss_ox = ('dissolved_oxygen','std'),
    perc_obs = ('bod',lambda x: (x>5).mean()*100)
).sort_values(by='perc_obs',ascending=False)

,mean_bod,median_nitrate,std_diss_ox,perc_obs
station,,,,
Hooghly_A,4.754002,3.461731,0.830046,37.5
Hooghly_B,4.329145,3.738032,1.373928,37.5
Damodar_B,3.841177,3.982679,0.894348,25.0
Rupnarayan,4.542133,4.243051,0.796034,25.0
Damodar_A,3.666460,3.561294,1.358965,12.5


#### ***Task 2 conclusion:***
Hoogly_A and Hoogly_B have highest percentage of observation with BOD greater than 5, at 37.5% each. This means these two stations experienced elevated BOD more frequently than the other stations in this dataset.

### *3. Detect high-risk observations*

In [36]:
water['risk_class'] = np.where(
    (
        (water['bod']>5).astype(int)+
        (water['nitrate']>5).astype(int)+
        (water['dissolved_oxygen']<5).astype(int)
    )>=2,
    'high_risk',
    'normal'
)
water.head()

,date,station,temperature,dissolved_oxygen,bod,nitrate,sampling_depth,month,risk_class
0,2025-01-01,Hooghly_A,30.987637,5.075960,3.525579,2.013784,1.301298,1,normal
1,2025-01-01,Hooghly_B,31.229921,6.026277,4.119309,4.899379,1.118234,1,normal
2,2025-01-01,Damodar_A,29.036414,8.267127,5.442308,3.561294,2.606949,1,normal
3,2025-01-01,Damodar_B,29.927131,6.026277,5.889212,5.843651,1.069208,1,high_risk
4,2025-01-01,Rupnarayan,31.178564,6.026277,4.182503,5.311404,2.394495,1,normal


In [35]:
risk_by_station = water.groupby('station').agg(
    total_obs = ('risk_class','size'),
    risk_count = ('risk_class', lambda x: (x == 'high_risk').sum())
)
risk_by_station['risk_perc'] = (risk_by_station['risk_count']/risk_by_station['total_obs'])*100
risk_by_station.sort_values(by='risk_perc',ascending=False)

,total_obs,risk_count,risk_perc
station,,,
Damodar_B,8,2,25.0
Rupnarayan,8,2,25.0
Hooghly_B,8,2,25.0
Damodar_A,8,0,0.0
Hooghly_A,8,0,0.0


In [40]:
risk_by_month = water.groupby('month').agg(
    total_obs = ('risk_class','size'),
    risk_count = ('risk_class', lambda x: (x =='high_risk').sum())
)
risk_by_month['risk_perc'] = (risk_by_month['risk_count']/risk_by_month['total_obs'])*100
risk_by_month.sort_values(by='risk_perc',ascending=False)

,total_obs,risk_count,risk_perc
month,,,
3,5,2,40.0
1,5,1,20.0
2,5,1,20.0
4,5,1,20.0
5,5,1,20.0
6,5,0,0.0
7,5,0,0.0
8,5,0,0.0


#### ***Task 3 conclusion:***
Damodar_B, Rupnarayan and Hoogly_B had highest proportion of high risk observations among the stations, with 25% each (2 out of 8 observations). Based on the defined risk criteria, March had the highest proportion of high risk observations among months, at 40% (2 out of 5 observations).